<a href="https://www.kaggle.com/code/samarthguptha/notebookc2efbb085d?scriptVersionId=352859383" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!apt-get update -qq
!apt-get install -y -qq colmap
!pip install -q open3d matplotlib opencv-python pillow tqdm

import os, sys, glob, subprocess
from PIL import Image
from tqdm import tqdm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:

In [2]:

RAW_DIR    = "/kaggle/input/datasets/samarthguptha/copy-of-lighthouse-survey-db4"
WORKSPACE  = "/kaggle/working/lighthouse_workspace_fast"
IMAGE_DIR  = f"{WORKSPACE}/images"
DB_PATH    = f"{WORKSPACE}/database.db"
SPARSE_DIR = f"{WORKSPACE}/sparse"
VOCAB_PATH = "/kaggle/working/vocab_tree_flickr100K_words32K.bin"

for d in [WORKSPACE, IMAGE_DIR, SPARSE_DIR]:
    os.makedirs(d, exist_ok=True)

NUM_THREADS = 4

def run_cmd(title, command):
    print(f"\n{'='*60}\n[*] {title}\n{'='*60}")
    command = f"stdbuf -oL -eL {command}"
    p = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True)
    for line in iter(p.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Step failed: {title}")
    print(f"[✓] {title} done")
raw_images = sorted(glob.glob(f"{RAW_DIR}/**/*.jpg", recursive=True))
print(f"Found {len(raw_images)} raw images")

print("Downscaling images...")
for img_path in tqdm(raw_images):
    dest = os.path.join(IMAGE_DIR, os.path.basename(img_path))
    with Image.open(img_path) as img:
        exif_data = img.info.get('exif')
        img.thumbnail((1800, 1800), Image.Resampling.LANCZOS)
        if exif_data:
            img.save(dest, "JPEG", quality=92, exif=exif_data)
        else:
            img.save(dest, "JPEG", quality=92)

print(f"[✓] {len(os.listdir(IMAGE_DIR))} images ready")

if not os.path.exists(VOCAB_PATH):
    print("Downloading vocab tree...")
    subprocess.run(f"wget -q https://demuc.de/colmap/vocab_tree_flickr100K_words32K.bin -O {VOCAB_PATH}", shell=True, check=True)
print(f"[✓] Vocab tree ready at {VOCAB_PATH}")

run_cmd(
    "SIFT Feature Extraction",
    f"colmap feature_extractor --database_path {DB_PATH} --image_path {IMAGE_DIR} "
    f"--ImageReader.single_camera 1 --SiftExtraction.use_gpu 0 --SiftExtraction.num_threads {NUM_THREADS}"
)



Found 200 raw images
Downscaling images...


100%|██████████| 200/200 [03:37<00:00,  1.09s/it]


[✓] 200 images ready
[✓] Vocab tree ready at /kaggle/working/vocab_tree_flickr100K_words32K.bin

[*] SIFT Feature Extraction

Feature extraction

Processed file [1/200]
  Name:            EX-01-10917_0283_0002.jpg
  Dimensions:      1800 x 1351
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1285.71px (Prior)
  GPS:             LAT=58.997, LON=10.780, ALT=101.455
  Features:        15393
Processed file [2/200]
  Name:            EX-01-10917_0283_0003.jpg
  Dimensions:      1800 x 1351
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1285.71px (Prior)
  GPS:             LAT=58.997, LON=10.780, ALT=101.265
  Features:        11577
Processed file [3/200]
  Name:            EX-01-10917_0283_0004.jpg
  Dimensions:      1800 x 1351
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    1285.71px (Prior)
  GPS:             LAT=58.997, LON=10.780, ALT=101.365
  Features:        13149
Processed file [4/200]
  Name:            EX-01-10917_0283_0005.jpg
  Dimensions:      1800 

In [3]:
run_cmd(
    "Vocab Tree Matching",
    f"colmap vocab_tree_matcher --database_path {DB_PATH} "
    f"--VocabTreeMatching.vocab_tree_path {VOCAB_PATH} "
    f"--VocabTreeMatching.num_images 20 "
    f"--SiftMatching.use_gpu 0 --SiftMatching.num_threads {NUM_THREADS}"
)


[*] Vocab Tree Matching

Vocabulary tree feature matching

Indexing image [1/200] in 0.579s
Indexing image [2/200] in 0.525s
Indexing image [3/200] in 0.537s
Indexing image [4/200] in 0.543s
Indexing image [5/200] in 0.530s
Indexing image [6/200] in 0.521s
Indexing image [7/200] in 0.499s
Indexing image [8/200] in 0.530s
Indexing image [9/200] in 0.585s
Indexing image [10/200] in 0.540s
Indexing image [11/200] in 0.545s
Indexing image [12/200] in 0.575s
Indexing image [13/200] in 0.620s
Indexing image [14/200] in 0.585s
Indexing image [15/200] in 0.584s
Indexing image [16/200] in 0.564s
Indexing image [17/200] in 0.573s
Indexing image [18/200] in 0.539s
Indexing image [19/200] in 0.602s
Indexing image [20/200] in 0.545s
Indexing image [21/200] in 0.579s
Indexing image [22/200] in 0.560s
Indexing image [23/200] in 0.623s
Indexing image [24/200] in 0.537s
Indexing image [25/200] in 0.562s
Indexing image [26/200] in 0.565s
Indexing image [27/200] in 0.539s
Indexing image [28/200] in 0.55

In [4]:
run_cmd(
    "Structure from Motion (SfM)",
    f"colmap mapper --database_path {DB_PATH} --image_path {IMAGE_DIR} --output_path {SPARSE_DIR} "
    f"--Mapper.ba_global_images_ratio 1.2 --Mapper.ba_global_points_ratio 1.2 --Mapper.num_threads {NUM_THREADS}"
)

print("\n SfM complete. Checking registration quality:")
!colmap model_analyzer --path {SPARSE_DIR}/0v



[*] Structure from Motion (SfM)

Loading database

Loading cameras... 1 in 0.000s
Loading matches... 1728 in 0.067s
Loading images... 200 in 0.184s (connected 200)
Building correspondence graph... in 0.553s (ignored 0)

Elapsed time: 0.013 [minutes]


Finding good initial image pair


Initializing with image pair #88 and #100


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  4.494499e+03    0.00e+00    2.83e+05   0.00e+00   0.00e+00  1.00e+04        0    1.55e-02    6.66e-02
   1  4.139143e+03    3.55e+02    1.06e+04   1.42e+00   1.00e+00  3.00e+04        1    3.32e-02    9.98e-02
   2  4.137749e+03    1.39e+00    8.48e+02   1.06e+00   1.04e+00  9.00e+04        1    2.74e-02    1.27e-01
   3  4.137074e+03    6.75e-01    3.37e+03   3.49e+00   1.01e+00  2.70e+05        1    2.74e-02    1.55e-01
   4  4.139047e+03   -1.97e+00    3.37e+03   9.89e+00  -8.45e-01  1.35e+05        1    1.34e-02    1.68

In [5]:
!ls -la {SPARSE_DIR}

total 12
drwxr-xr-x 3 root root 4096 Sep 26 05:21 .
drwxr-xr-x 4 root root 4096 Sep 26 04:01 ..
drwxr-xr-x 2 root root 4096 Sep 26 05:21 0


In [6]:
!colmap model_analyzer --path {SPARSE_DIR}/0

Cameras: 1
Images: 200
Registered images: 200
Points: 148208
Observations: 659523
Mean track length: 4.449982
Mean observations per image: 3297.615000
Mean reprojection error: 1.221544px


In [7]:
run_cmd(
    "Export sparse model to PLY",
    f"colmap model_converter --input_path {SPARSE_DIR}/0 --output_path {WORKSPACE}/sparse_point_cloud.ply --output_type PLY"
)


[*] Export sparse model to PLY
[✓] Export sparse model to PLY done
